# Pipeline ETL — E-commerce

Neste notebook executamos o pipeline ETL (Extract, Transform, Load):
- **Extract**: leitura dos CSVs gerados
- **Transform**: limpeza e conversão de tipos
- **Load**: carga no banco SQLite e criação das views analíticas

## 1. Importações e Configurações

In [1]:
import pandas as pd
import sqlite3
import os

RAW_DIR = "data/raw"
DB_PATH = "data/database.db"

conn = sqlite3.connect(DB_PATH)
print("Conexão com o banco estabelecida!")

Conexão com o banco estabelecida!


## 2. Extract — Carregando os CSVs

In [ ]:
clientes  = pd.read_csv(f"{RAW_DIR}/clientes.csv")
produtos  = pd.read_csv(f"{RAW_DIR}/produtos.csv")
pedidos   = pd.read_csv(f"{RAW_DIR}/pedidos.csv")
itens     = pd.read_csv(f"{RAW_DIR}/itens_pedido.csv")

print(f"Clientes:  {len(clientes)} linhas")
print(f"Produtos:  {len(produtos)} linhas")
print(f"Pedidos:   {len(pedidos)} linhas")
print(f"Itens:     {len(itens)} linhas")

Clientes:  500 linhas
Produtos:  80 linhas
Pedidos:   3000 linhas
Itens:     5719 linhas


## 3. Transform — Limpeza e Conversão de Tipos

Convertemos as colunas de data para o tipo correto e verificamos a existência de valores nulos.

In [3]:
clientes["data_cadastro"] = pd.to_datetime(clientes["data_cadastro"])
pedidos["data_pedido"]    = pd.to_datetime(pedidos["data_pedido"])

print("Nulos em pedidos:")
print(pedidos.isnull().sum())

Nulos em pedidos:
pedido_id               0
cliente_id              0
data_pedido             0
hora_pedido             0
status                  0
forma_pagamento         0
frete                   0
desconto_pct            0
data_entrega         1152
canal_venda             0
subtotal_produtos       0
desconto_valor          0
valor_total             0
dtype: int64


Os 1.152 nulos em `data_entrega` são esperados — pedidos cancelados ou em processamento não possuem data de entrega.

## 4. Load — Carga no Banco SQLite

In [4]:
clientes.to_sql("clientes",    conn, if_exists="replace", index=False)
produtos.to_sql("produtos",    conn, if_exists="replace", index=False)
pedidos.to_sql("pedidos",      conn, if_exists="replace", index=False)
itens.to_sql("itens_pedido",   conn, if_exists="replace", index=False)

print("Tabelas carregadas no banco!")

Tabelas carregadas no banco!


## 5. Criando as Views Analíticas

In [7]:
with open("../sql/01_create_views.sql", "r", encoding="utf-8") as f:
    sql = f.read()

conn.executescript(sql)
print("Views criadas!")

# Testando uma view
pd.read_sql("SELECT * FROM vw_faturamento_mensal LIMIT 5", conn)

Views criadas!


,ano_mes,total_pedidos,faturamento,ticket_medio
0,2024-06,72,132717.87,1843.30
1,2024-07,61,179387.70,2940.78
2,2024-08,99,298061.77,3010.72
3,2024-09,97,249289.73,2570.00
4,2024-10,91,224000.75,2461.55


## 6. Exportando as Views para CSV

Exportamos as views para CSV para uso no Power BI.

In [8]:
os.makedirs("data/processed", exist_ok=True)

views = [
    "vw_faturamento_mensal",
    "vw_top_produtos",
    "vw_vendas_por_regiao",
    "vw_vendas_por_canal",
]

for view in views:
    df = pd.read_sql(f"SELECT * FROM {view}", conn)
    df.to_csv(f"data/processed/{view}.csv", index=False, encoding="utf-8")
    print(f"✓ {view}.csv exportado")

conn.close()
print("\nETL concluído!")

✓ vw_faturamento_mensal.csv exportado
✓ vw_top_produtos.csv exportado
✓ vw_vendas_por_regiao.csv exportado
✓ vw_vendas_por_canal.csv exportado

ETL concluído!
